# imports 

In [2]:
import os
import pandas as pd

import mlflow
import mlflow.pyfunc

# Models
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor


# Class 

In [47]:
import os
import mlflow
import mlflow.pyfunc
import pandas as pd
import json
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler


class MLflowLoadModels:
    """
    Класс для загрузки последней модели и скейлера из MLflow, а также выполнения предсказаний.
    
    Методы:
    - __init__: Инициализация объекта класса и настройка MLflow.
    - _get_latest_model_name: Определение имени модели с последней версией.
    - _get_latest_version: Определение последней версии модели.
    - load_model: Загрузка модели.
    - load_scaler: Загрузка скейлера из артефактов модели.
    """
    
    def __init__(self, tracking_uri: str|None = None):
        """
        Инициализирует MLflowModels, настраивает URI отслеживания MLflow и создает клиент.

        :param tracking_uri: URI MLflow Tracking Server.
        """
        if tracking_uri is None:
            tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
            if tracking_uri is None:
                raise ValueError("Переменная окружения 'MLFLOW_TRACKING_URI' не найдена, tracking_uri не может быть автоматически определен.")

        mlflow.set_tracking_uri(tracking_uri)
        self.__client = mlflow.tracking.MlflowClient()
        self.__model_name = None
        self.__latest_version = None

    def _get_latest_model_name(self) -> str:
        """
        Возвращает имя модели с самой последней версией по времени создания версии.

        :return: Имя модели.
        """
        models = self.__client.search_registered_models()
        
        if not models:
            raise ValueError("В Model Registry нет зарегистрированных моделей.")
        
        latest_model = None
        latest_timestamp = None
        
        for model in models:
            for version in model.latest_versions:
                if latest_timestamp is None or version.creation_timestamp > latest_timestamp:
                    latest_timestamp = version.creation_timestamp
                    latest_model = model.name
        
        if latest_model is None:
            raise ValueError("Не удалось найти последнюю модель.")
        
        return latest_model
    
    def _get_latest_version(self) -> str:
        """
        Возвращает последнюю версию модели по имени модели.

        :return: Последняя версия модели.
        """
        versions = self.__client.search_model_versions(f"name='{self.__model_name}'")
        
        if not versions:
            raise ValueError(f"Для модели '{self.__model_name}' нет доступных версий.")
        
        return max(versions, key=lambda v: v.creation_timestamp).version
    
    def _ensure_model_and_version(self, model_name: str|None, version: str|None):
        """
        Проверяет и загружает имя модели и версию, если они не были заданы.

        :param model_name: Имя модели.
        :param version: Версия модели.
        """
        if model_name is None:
            if self.__model_name is None:
                self.__model_name = self._get_latest_model_name()
        else:
            self.__model_name = model_name

        if version is None:
            if self.__latest_version is None:
                self.__latest_version = self._get_latest_version()
        else:
            self.__latest_version = version

    def load_model(self, model_name: str|None = None, version: str|None = None) -> mlflow.pyfunc.PyFuncModel:
        """
        Загружает модель по имени модели и версии.

        :param model_name: Имя модели.
        :param version: Версия модели.
        :return: Загруженная модель.
        """
        self._ensure_model_and_version(model_name, version)
        return mlflow.pyfunc.load_model(f"models:/{self.__model_name}/{self.__latest_version}")
    
    def load_scaler(self, model_name: str|None = None, version: str|None = None) -> object:
        """
        Загружает скейлер из артефактов модели.

        :param model_name: Имя модели.
        :param version: Версия модели.
        :return: Загруженный скейлер.
        """
        self._ensure_model_and_version(model_name, version)
        
        # Загружаем артефакты модели
        artifact_uri = self.__client.get_model_version(self.__model_name, self.__latest_version).source

        if artifact_uri.endswith("/model"):
            artifact_uri = os.path.dirname(artifact_uri)

        # Правильный путь к файлу scaler_param.json
        scaler_path = os.path.join(artifact_uri, "scaler_param.json")

        # Проверка наличия файла
        if not os.path.exists(scaler_path):
            raise FileNotFoundError(f"Файл 'scaler_param.json' не найден в артефактах модели.")
        
        # Загружаем скейлер
        with open(scaler_path, 'r') as f:
            scaler_params = json.load(f)

        scaler_type = scaler_params.get('type', 'StandardScaler')  # Значение по умолчанию
            
        if scaler_type == 'StandardScaler':
            scaler = StandardScaler()
        elif scaler_type == 'MinMaxScaler':
            scaler = MinMaxScaler()
        elif scaler_type == 'RobustScaler':
            scaler = RobustScaler()
        else:
            raise ValueError(f"Неизвестный тип скейлера: {scaler_type}")
        
        if 'mean' in scaler_params:
            scaler.mean_ = scaler_params['mean']
        if 'scale' in scaler_params:
            scaler.scale_ = scaler_params['scale']
        if 'var' in scaler_params:
            scaler.var_ = scaler_params['var']
        if 'min' in scaler_params:
            scaler.min_ = scaler_params['min']
        if 'data_min' in scaler_params:
            scaler.data_min_ = scaler_params['data_min']
        if 'data_max' in scaler_params:
            scaler.data_max_ = scaler_params['data_max']
        if 'center' in scaler_params:
            scaler.center_ = scaler_params['center']
        if 'quantiles' in scaler_params:
            scaler.quantiles_ = scaler_params['quantiles']
        
        return scaler


# Пример использования класса
if __name__ == "__main__":
    # Создаем объект класса
    models = MLflowLoadModels()
    predict_model = models.load_model()
    scaler = models.load_scaler()
    